# Network analysis

## Transition to giant component

Depending on the threshold distance chosen to build the adjacency matrix, we have different networks. We saw that with threshold equal to 0.01 we have a connected network, while with 0.001 we have 14986 components. We want a network with the right number of connected components (almost equal to the number of topics of papers), so we build adjacency matrices sweeping thesholds from 0.001 to 0.01.

First, however, we need to know the number of topics we have.

We have 22 different topics, and we can see that with 0.005 threshold we have 11 components, and with 0.004 we have 196 components. So the right threshold should be between 0.004 and 0.005.

Actually for 0.005 we have one giant component with all papers and 10 disconnected papers, so maybe better to use 0.004. Also 0.004 has only one giant component. We need to find the correct value in which we have the right number of components.

In [34]:
import pandas as pd
from networks.build_networks import build_adjacency_matrix
import networkx as nx
from collections import Counter

In [35]:
#take the list of all paper topics, putting the primary cathegory if this is a cathegory of physics, otherwise the secondary cathegory
all_papers = pd.read_csv("data/all_papers.csv")
all_physics_topics = ["Accelerator Physics", "Applied Physics", "Atmospheric and Oceanic Physics", "Atomic and Molecular Clusters", 
                      "Atomic Physics", "Biological Physics", "Chemical Physics", "Classical Physics", "Computational Physics", 
                      "Data Analysis, Statistics and Probability", "Fluid Dynamics", "General Physics", "Geophysics", 
                      "History and Philosophy of Physics", "Instrumentation and Detectors", "Medical Physics", "Optics",
                        "Physics and Society", "Physics Education", "Plasma Physics", "Popular Physics", "Space Physics"]

In [36]:
topics_list = [all_papers['primary_cathegory'][i] if all_papers['primary_cathegory'][i] in all_physics_topics 
               else all_papers['secondary_cathegory'][i] for i in range(25877)]

Now create the network using threshold 0.005 and assign the attributes to the nodes.

In [52]:
abstract_network = nx.from_scipy_sparse_array(build_adjacency_matrix(threshold = 0.002))
nx.number_connected_components(abstract_network)

5715

In [ ]:
del all_papers

Create now a dictionary of paper topics, where the keys are progressive integers (one for each paper) and values the topics of those papers.

In [53]:
keys = list(range(25877))
topic_for_each_paper = dict(zip(keys, topics_list))

In [54]:
nx.set_node_attributes(abstract_network, topic_for_each_paper, name = "Primary topic")

In [55]:
abstract_network.nodes[10]

{'Primary topic': 'Physics and Society'}

Now try to see if we can recover the number of papers for each topic

In [56]:
Counter(nx.get_node_attributes(abstract_network, "Primary topic").values())

Counter({'Optics': 4587,
         'Fluid Dynamics': 2815,
         'Chemical Physics': 2216,
         'Computational Physics': 2134,
         'Applied Physics': 1929,
         'Instrumentation and Detectors': 1572,
         'Physics and Society': 1522,
         'Plasma Physics': 1409,
         'Atomic Physics': 1355,
         'Biological Physics': 1062,
         'Medical Physics': 967,
         'Atmospheric and Oceanic Physics': 772,
         'Geophysics': 636,
         'Data Analysis, Statistics and Probability': 513,
         'Accelerator Physics': 486,
         'Classical Physics': 394,
         'Physics Education': 367,
         'Space Physics': 365,
         'History and Philosophy of Physics': 329,
         'General Physics': 284,
         'Atomic and Molecular Clusters': 91,
         'Popular Physics': 72})

## Try giant component transition function

By using this function we can notice that we ALWAYS have a giant component followed by a series of detached papers, not connected to any other. So, we expected to have papers clustered following the different topics, but what happens is completely different. The next step is to create a fully connected network and study its property: the analogy with a null model (maybe random), and the presence of communities inside the network. So we will use a threshold of 0.01.

WRITE THIS IN THE REPORT.

## Comparison with null random model: degree distribution

In [1]:
import pandas as pd
from networks.build_networks import build_adjacency_matrix
import networkx as nx
from collections import Counter
import numpy as np
from scipy.sparse import load_npz

In [2]:
#load network
adjacency_matrix = load_npz("networks/abstract_tfidf_adjacency_0_01.npz")
G = nx.from_scipy_sparse_array(adjacency_matrix)

In [5]:
nx.number_connected_components(G)

1

In [7]:
#calculate degree by summing rows (or columns, it is symmetric) of adjacency matrix
degree_distribution = adjacency_matrix.sum(axis = 0)

In [8]:
data = pd.DataFrame(data = degree_distribution, columns = ["Degree"])

In [10]:
data.to_csv("networks/results/degree_distribution.csv")

In [14]:
print(np.mean(degree_distribution))
print(np.median(degree_distribution))

40.357846736484134
12.0


Differently from what expected, the distribution of degrees is not a Gaussian: the random model can be easily rejected. The distribution is completely right skewed: the difference between the mean and the median clearly shows that we have a long-tail asymmetric distribution.